In [8]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
import os

In [4]:
df = pd.read_csv('data_untuk_modeling.csv')
df.head()

,Unnamed: 0,Judul,Industri,Tipe,Kategori,Kota,Provinsi,Negara,Gaji_Min,Gaji_Max,Skills,Pengalaman_Bulan,Pendidikan,Deskripsi
0,0,IT Business Analyst,Information Technology and Services,FULL_TIME,System Analyst,Jakarta Timur,DKI Jakarta,ID,Tidak ada,Tidak ada,"SQL, Business Analysis, Relational Databases, ...",36.0,bachelor degree,Job description \n \n Evaluating and documenti...
1,1,IT System Analyst,Information Technology and Services,FULL_TIME,System Analyst,Surabaya,Jawa Timur,ID,7000000.0,9000000.0,"Requirements Analysis, SQL, Business Analysis,...",12.0,bachelor degree,General Qualification : \n \n Man (23 - 35 yea...
2,2,IT Business Analyst - Insurance exp,Information Technology and Services,CONTRACTOR,Business Analyst,Jakarta Selatan,DKI Jakarta,ID,13000000.0,14500000.0,"SDLC, Life Insurance, Teamwork, Business Proce...",36.0,bachelor degree,Skill And Experience : \n - Understand the Bas...
3,3,IT Business Analyst (for Insurance industry),Information Technology and Services,CONTRACTOR,Business Analyst,Jakarta Selatan,DKI Jakarta,ID,5000000.0,7500000.0,"Csm, SQL, Insurance, Group Asia, SDLC, Life 40...",36.0,bachelor degree,Understand the Basic Life Insurance business r...
4,4,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada,Tidak ada


In [5]:
daftar_judul = df['Judul'].tolist()
daftar_judul[:5]

['IT Business Analyst',
 'IT System Analyst',
 'IT Business Analyst - Insurance exp',
 'IT Business Analyst (for Insurance industry)',
 'Tidak ada']

In [7]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_vectorizer.fit(daftar_judul)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [9]:
save_path = os.path.join('model_hasil', 'tfidf_fitted.pkl')
with open(save_path, 'wb') as f:
        pickle.dump(tfidf_vectorizer, f)
        
print(f"🎉 Sukses! File TF-IDF berhasil disimpan di: {save_path}")

🎉 Sukses! File TF-IDF berhasil disimpan di: model_hasil\tfidf_fitted.pkl


In [11]:
import tensorflow as tf

# 1. Load model .keras milikmu
model = tf.keras.models.load_model('./model_hasil/model_rekomendasi_3.keras')

# 2. Inisiasi Converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Operasi bawaan TFLite
    tf.lite.OpsSet.SELECT_TF_OPS    # Operasi bawaan TensorFlow (Wajib untuk LSTM)
]
# Matikan penurunan (lowering) eksperimental untuk TensorList
converter._experimental_lower_tensor_list_ops = False
# ========================================================

# 4. Opsional tapi disarankan: Aktifkan optimasi
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. Lakukan konversi
tflite_model = converter.convert()

# 4. Simpan hasilnya
with open('./model_hasil/model_rekomendasi_3.tflite', 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\andre\AppData\Local\Temp\tmpdnbjym6p\assets


INFO:tensorflow:Assets written to: C:\Users\andre\AppData\Local\Temp\tmpdnbjym6p\assets


Saved artifact at 'C:\Users\andre\AppData\Local\Temp\tmpdnbjym6p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 299), dtype=tf.float32, name='cv_input'), TensorSpec(shape=(None, 299), dtype=tf.float32, name='job_input')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2932362962576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362961616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362964880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362960464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362963152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362964688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362961232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362961424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2932362959696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  293236296104

### Mencoba pendekatan ONNX

In [12]:
import tensorflow as tf
import tf2onnx
import onnx

# 1. Load model Keras-mu
model = tf.keras.models.load_model('./model_hasil/model_rekomendasi_3.keras')

MAX_LEN = 299

# 2. Definisikan "pintu masuk" (input signature) model
# Nilai 'None' pada dimensi pertama berarti ONNX akan menerima jumlah data (batch) berapapun secara dinamis!
spec = (
    tf.TensorSpec((None, MAX_LEN), tf.float32, name="input_cv"),
    tf.TensorSpec((None, MAX_LEN), tf.float32, name="input_job")
)

# 3. Lakukan konversi (Opset 13 adalah versi standar yang sangat stabil untuk LSTM)
output_path = "./model_hasil/model_rekomendasi_3.onnx"
model_proto, _ = tf2onnx.convert.from_keras(
    model, 
    input_signature=spec, 
    opset=13, 
    output_path=output_path
)

print("🎉 Berhasil! File model_rekomendasi_3.onnx siap digunakan.")

🎉 Berhasil! File model_rekomendasi_3.onnx siap digunakan.


### PKL ternyata masih butuh keras, kita ubah ke JSON

In [13]:
import pickle
import json

# 1. Load Tokenizer lama menggunakan Keras/Pickle
print("Membaca file pickle lama...")
with open('./model_hasil/tokenizer_loker3.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

# 2. Ekstrak data mentahnya ke dalam Dictionary Python biasa
tokenizer_data = {
    'word_index': tokenizer.word_index,
    'oov_token': tokenizer.oov_token
}

# 3. Simpan sebagai file JSON yang ringan dan aman
with open('./model_hasil/tokenizer_dict3.json', 'w') as f:
    json.dump(tokenizer_data, f)

print("🎉 Sukses! File tokenizer_dict.json berhasil dibuat.")

Membaca file pickle lama...
🎉 Sukses! File tokenizer_dict.json berhasil dibuat.
